In [8]:
# ============================================================
# CELL 1 — Load Saved Artifacts and Reconstruct Test Set
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from pathlib import Path
import sys

# Works on any machine — go up from notebooks/ to project root
BASE = Path.cwd().parent
sys.path.insert(0, str(BASE))

# Load saved model artifacts
lr     = joblib.load(f'{BASE}/models/lead_scoring_model.pkl')
scaler = joblib.load(f'{BASE}/models/scaler.pkl')
model_cols = joblib.load(f'{BASE}/models/model_columns.pkl')

print(f"Model loaded: {type(lr).__name__}")
print(f"Features expected: {len(model_cols)}")

# Reconstruct the exact same preprocessing pipeline
df = pd.read_csv(str(BASE / 'data' / 'Leads.csv'))
df_clean = df.copy()

drop_cols = [
    'Prospect ID', 'Lead Number', 'Lead Quality',
    'Asymmetrique Activity Index', 'Asymmetrique Profile Index',
    'How did you hear about X Education',
    'I agree to pay the amount through cheque',
    'Get updates on DM Content', 'Update me on Supply Chain Content',
    'Receive More Updates About Our Courses'
]
df_clean.drop(columns=drop_cols, inplace=True)

for col in ['TotalVisits', 'Page Views Per Visit',
            'Asymmetrique Activity Score', 'Asymmetrique Profile Score']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

binary_cols = [
    'Do Not Email', 'Do Not Call', 'Search', 'Magazine',
    'Newspaper Article', 'X Education Forums', 'Newspaper',
    'Digital Advertisement', 'Through Recommendations',
    'A free copy of Mastering The Interview'
]
for col in binary_cols:
    df_clean[col] = df_clean[col].fillna('No')

cat_cols = [
    'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'What is your current occupation',
    'What matters most to you in choosing a course',
    'Tags', 'Lead Profile', 'City', 'Last Notable Activity'
]
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

df_clean = df_clean.replace('Select', np.nan)
for col in df_clean.select_dtypes(include='object').columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

def response_time_bucket(time):
    if time == 0: return 'No Engagement'
    elif time <= 300: return 'Low (0-5 min)'
    elif time <= 900: return 'Medium (5-15 min)'
    else: return 'High (15+ min)'

df_clean['Engagement_Level'] = df_clean['Total Time Spent on Website'].apply(response_time_bucket)
df_clean['High_Interaction'] = (df_clean['TotalVisits'] >= 5).astype(int)

def group_lead_source(source):
    if source in ['Google', 'Organic Search', 'Bing']: return 'Search Engine'
    elif source in ['Direct Traffic', 'NC_EDM']: return 'Direct'
    elif source in ['Olark Chat', 'Live Chat']: return 'Chat'
    elif source in ['WeLearn', 'Welingak Website']: return 'Partner'
    elif source in ['Reference', 'Through Recommendations']: return 'Referral'
    elif source in ['Social Media', 'Facebook', 'youtube']: return 'Social'
    else: return 'Other'

df_clean['Lead_Source_Group'] = df_clean['Lead Source'].apply(group_lead_source)
df_clean['Is_Webinar_Lead'] = (df_clean['Lead Source'] == 'Webinar').astype(int)
df_clean['Engagement_Score'] = (
    df_clean['Total Time Spent on Website'] * 0.5 +
    df_clean['TotalVisits'] * 10 +
    df_clean['Page Views Per Visit'] * 5
)

for col in binary_cols:
    df_clean[col] = df_clean[col].map({'Yes': 1, 'No': 0})

engagement_order = {
    'No Engagement': 0, 'Low (0-5 min)': 1,
    'Medium (5-15 min)': 2, 'High (15+ min)': 3
}
df_clean['Engagement_Level'] = df_clean['Engagement_Level'].map(engagement_order)

ohe_cols = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country',
    'Specialization', 'What is your current occupation',
    'What matters most to you in choosing a course',
    'Tags', 'Lead Profile', 'City', 'Lead_Source_Group',
    'Last Notable Activity'
]
df_encoded = pd.get_dummies(df_clean, columns=ohe_cols, drop_first=True)

X = df_encoded.drop(columns=['Converted'])
y = df_encoded['Converted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

X_test_scaled = scaler.transform(X_test)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

print(f"Test set: {len(X_test)} leads")
print(f"Probabilities generated: {len(lr_probs)}")
print("Ready for Phase 5 and Phase 6 analysis.")

Model loaded: LogisticRegression
Features expected: 175
Test set: 1848 leads
Probabilities generated: 1848
Ready for Phase 5 and Phase 6 analysis.


In [9]:
# ============================================================
# CELL 2 — PHASE 5: Baseline vs ML Pipeline Comparison
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

print("=" * 60)
print("PHASE 5 — BASELINE vs ML PIPELINE COMPARISON")
print("=" * 60)

# Align raw data to test set indices
raw = pd.read_csv(str(BASE / 'data' / 'Leads.csv'))
raw_test = raw.iloc[X_test.index].reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

# ── Baseline Rule System ──
def baseline_predict(row):
    if str(row.get('Lead Source', '')).strip() == 'Webinar':
        return 1
    if pd.to_numeric(row.get('TotalVisits', 0), errors='coerce') >= 5:
        return 1
    return 0

baseline_preds = raw_test.apply(baseline_predict, axis=1)

# ── ML Pipeline predictions at 0.50 threshold ──
lr_preds = (lr_probs >= 0.50).astype(int)

# ── Comparison Table ──
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC'],
    'Baseline (Rules)': [
        round(accuracy_score(y_test_reset, baseline_preds), 4),
        round(precision_score(y_test_reset, baseline_preds, zero_division=0), 4),
        round(recall_score(y_test_reset, baseline_preds, zero_division=0), 4),
        round(f1_score(y_test_reset, baseline_preds, zero_division=0), 4),
        'N/A'
    ],
    'ML Pipeline (LR)': [
        round(accuracy_score(y_test, lr_preds), 4),
        round(precision_score(y_test, lr_preds, zero_division=0), 4),
        round(recall_score(y_test, lr_preds, zero_division=0), 4),
        round(f1_score(y_test, lr_preds, zero_division=0), 4),
        round(roc_auc_score(y_test, lr_probs), 4)
    ],
})

print(comparison.to_string(index=False))

# ── Business Impact ──
total_neg = (y_test.values == 0).sum()
deflection_baseline = ((baseline_preds == 0) & (y_test_reset.values == 0)).sum()
deflection_ml = ((lr_preds == 0) & (y_test.values == 0)).sum()

bl_rt = [1 if p == 1 else 168 for p in baseline_preds]
lr_rt = [1 if p > 0.75 else 8 if p >= 0.50 else 168 for p in lr_probs]

print(f"\n── BUSINESS IMPACT ──")
print(f"Leads flagged by Baseline Rule 1 (Webinar): {(raw_test['Lead Source'] == 'Webinar').sum()}")
print(f"Leads flagged by Baseline Rule 2 (5+ visits): {(raw_test['TotalVisits'] >= 5).sum()}")
print(f"\nDeflection Rate - Baseline   : {deflection_baseline/total_neg*100:.1f}%")
print(f"Deflection Rate - ML Pipeline: {deflection_ml/total_neg*100:.1f}%")
print(f"\nAvg Lead Response Time - Baseline   : {np.mean(bl_rt):.1f} hours")
print(f"Avg Lead Response Time - ML Pipeline: {np.mean(lr_rt):.1f} hours")
print(f"Response Time Improvement           : {((np.mean(bl_rt)-np.mean(lr_rt))/np.mean(bl_rt)*100):.1f}%")
print("\nPhase 5 Complete.")

PHASE 5 — BASELINE vs ML PIPELINE COMPARISON
   Metric Baseline (Rules)  ML Pipeline (LR)
 Accuracy           0.5893            0.8907
Precision           0.4521            0.8622
   Recall           0.3118            0.8525
 F1 Score           0.3691            0.8573
  AUC-ROC              N/A            0.9510

── BUSINESS IMPACT ──
Leads flagged by Baseline Rule 1 (Webinar): 0
Leads flagged by Baseline Rule 2 (5+ visits): 491

Deflection Rate - Baseline   : 76.3%
Deflection Rate - ML Pipeline: 91.5%

Avg Lead Response Time - Baseline   : 123.6 hours
Avg Lead Response Time - ML Pipeline: 104.8 hours
Response Time Improvement           : 15.2%

Phase 5 Complete.


In [4]:
# ============================================================
# CELL 3 — PHASE 6: Batch Agent Simulation & Observability
# ============================================================

print("=" * 60)
print("PHASE 6 — AGENT SIMULATION ACROSS ALL TEST LEADS")
print("=" * 60)

def agent_action(score):
    if score > 0.75:    return "Route to Sales Rep"
    elif score >= 0.55: return "Email Sequence"
    elif score >= 0.45: return "HITL Escalation"
    else:               return "Nurture + Deflect"

sim_df = pd.DataFrame({
    'Lead_Score':        lr_probs,
    'Actual_Converted':  y_test.values,
    'Agent_Action':      [agent_action(p) for p in lr_probs]
})

# ── 1. Action Distribution ──
print("\n── ACTION DISTRIBUTION ──")
for action, count in sim_df['Agent_Action'].value_counts().items():
    print(f"  {action}: {count} leads ({count/len(sim_df)*100:.1f}%)")

# ── 2. Deflection Rate ──
deflected = (sim_df['Agent_Action'] == 'Nurture + Deflect').sum()
correct_deflections = ((sim_df['Agent_Action'] == 'Nurture + Deflect') &
                       (sim_df['Actual_Converted'] == 0)).sum()
print(f"\n── DEFLECTION RATE ──")
print(f"  Leads deflected: {deflected} ({deflected/len(sim_df)*100:.1f}%)")
print(f"  Correctly deflected: {correct_deflections} ({correct_deflections/deflected*100:.1f}% accuracy)")

# ── 3. Threshold Adherence Score ──
sim_df['Expected_Action'] = sim_df['Lead_Score'].apply(agent_action)
adherent = (sim_df['Agent_Action'] == sim_df['Expected_Action']).sum()
adherence_score = adherent / len(sim_df) * 100
print(f"\n── THRESHOLD ADHERENCE SCORE ──")
print(f"  Actions matching threshold logic: {adherent}/{len(sim_df)}")
print(f"  Threshold Adherence Score: {adherence_score:.2f}%")

# ── 4. Action Hallucination Rate ──
hallucinations = len(sim_df) - adherent
hallucination_rate = hallucinations / len(sim_df) * 100
print(f"\n── ACTION HALLUCINATION RATE ──")
print(f"  Boundary violations: {hallucinations}")
print(f"  Action Hallucination Rate: {hallucination_rate:.2f}%")

# ── 5. False Positive Rate in High Priority Routing ──
high_priority = sim_df[sim_df['Agent_Action'] == 'Route to Sales Rep']
false_positives = (high_priority['Actual_Converted'] == 0).sum()
fpr_high = false_positives / len(high_priority) * 100
print(f"\n── FALSE POSITIVE RATE (HIGH PRIORITY ROUTING) ──")
print(f"  Leads routed to Sales Rep: {len(high_priority)}")
print(f"  Did NOT convert (false positives): {false_positives}")
print(f"  False Positive Rate: {fpr_high:.1f}%")
print(f"  Precision at 0.75 threshold: {100-fpr_high:.1f}%")

# ── 6. Lead Response Time ──
response_times = [1 if p > 0.75 else 8 if p >= 0.55
                  else 24 if p >= 0.45 else 168 for p in lr_probs]
print(f"\n── LEAD RESPONSE TIME SIMULATION ──")
print(f"  Average Lead Response Time: {np.mean(response_times):.1f} hours")

# ── 7. HITL Band Analysis ──
hitl = sim_df[sim_df['Agent_Action'] == 'HITL Escalation']
hitl_convert_rate = hitl['Actual_Converted'].mean() * 100
print(f"\n── HITL BAND ANALYSIS ──")
print(f"  Leads escalated to human review: {len(hitl)}")
print(f"  Actual conversion rate in HITL band: {hitl_convert_rate:.1f}%")
print(f"  (Validates HITL band as genuinely uncertain zone)")

# ── 8. Summary ──
print(f"\n{'='*60}")
print("PHASE 6 OBSERVABILITY SUMMARY")
print(f"{'='*60}")
print(f"  Total test leads processed   : {len(sim_df)}")
print(f"  Deflection Rate              : {deflected/len(sim_df)*100:.1f}%")
print(f"  Threshold Adherence Score    : {adherence_score:.2f}%")
print(f"  Action Hallucination Rate    : {hallucination_rate:.2f}%")
print(f"  False Positive Rate (High)   : {fpr_high:.1f}%")
print(f"  Avg Lead Response Time       : {np.mean(response_times):.1f} hours")

# Save results
sim_df.to_csv(f'{BASE}/outputs/phase6_agent_simulation.csv', index=False)
print(f"\nResults saved to outputs/phase6_agent_simulation.csv")
print("\nPhase 6 Complete.")

PHASE 6 — AGENT SIMULATION ACROSS ALL TEST LEADS

── ACTION DISTRIBUTION ──
  Nurture + Deflect: 1115 leads (60.3%)
  Route to Sales Rep: 583 leads (31.5%)
  Email Sequence: 107 leads (5.8%)
  HITL Escalation: 43 leads (2.3%)

── DEFLECTION RATE ──
  Leads deflected: 1115 (60.3%)
  Correctly deflected: 1024 (91.8% accuracy)

── THRESHOLD ADHERENCE SCORE ──
  Actions matching threshold logic: 1848/1848
  Threshold Adherence Score: 100.00%

── ACTION HALLUCINATION RATE ──
  Boundary violations: 0
  Action Hallucination Rate: 0.00%

── FALSE POSITIVE RATE (HIGH PRIORITY ROUTING) ──
  Leads routed to Sales Rep: 583
  Did NOT convert (false positives): 52
  False Positive Rate: 8.9%
  Precision at 0.75 threshold: 91.1%

── LEAD RESPONSE TIME SIMULATION ──
  Average Lead Response Time: 102.7 hours

── HITL BAND ANALYSIS ──
  Leads escalated to human review: 43
  Actual conversion rate in HITL band: 53.5%
  (Validates HITL band as genuinely uncertain zone)

PHASE 6 OBSERVABILITY SUMMARY
  Tot